# NumPy Lab — AI Lab Instructor Notebook

*Python & Data · Easy*

Build deep intuition for NumPy internals, vectorization, and performance — the way FAANG expects ML engineers to think.

**Outcome.** You can write fast, memory-efficient, interview-ready NumPy code and explain *why* it is efficient.

15 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
import time

def check(name, condition):
    if bool(condition):
        print(f"[PASS] {name}")
    else:
        raise AssertionError(f"[FAIL] {name}")


## Task 1: Dataset & Setup

Setup

Section 1 — ndarray Fundamentals

Task 1.1: Array Creation & Shapes

In [ ]:
import numpy as np
import time

## Task 2: Array Creation & Shapes
*Section: ndarray Fundamentals*

Section 1 — ndarray Fundamentals

Task 1.1: Array Creation & Shapes

**Hints**
- # - Use np.arange
- # - reshape does NOT copy data
- # - Total elements must match

In [ ]:
# Task 1.1 Solutions
arr_1d = np.arange(100)
arr_2d = arr_1d.reshape(10, 10)
arr_3d = np.arange(4 * 5 * 3).reshape(4, 5, 3)

assert arr_1d.shape == (100,)
assert arr_2d.shape == (10, 10)
assert arr_3d.shape == (4, 5, 3)

print(arr_1d[:5], arr_1d[-5:])
print(arr_2d.shape, arr_2d[0, :5])
print(arr_3d.shape, arr_3d[0, 0, :])

In [ ]:
# Checks
check('arr_1d_shape', isinstance(arr_1d, np.ndarray) and arr_1d.shape == (100,))
check('arr_2d_shape', isinstance(arr_2d, np.ndarray) and arr_2d.shape == (10, 10))
check('arr_3d_shape', isinstance(arr_3d, np.ndarray) and arr_3d.shape == (4, 5, 3))
check('arr_2d_is_view_like', np.shares_memory(arr_1d, arr_2d))

## Task 3: Compare memory usage
*Section: dtype & Memory*

**Explain:**
- What does `.shape` represent?
- Why does contiguous memory matter?

Section 1.2 — dtype & Memory

Task 1.2: Compare memory usage

**Hints**
- # - Use .nbytes

In [ ]:
base = np.arange(1000)
arr_int = base.astype(np.int64)
arr_float = base.astype(np.float64)

print('int64 nbytes:', arr_int.nbytes)
print('float64 nbytes:', arr_float.nbytes)
print('dtype:', arr_int.dtype, arr_float.dtype)

In [ ]:
# Checks
check('same_shape', arr_int.shape == arr_float.shape)
check('dtype_int', np.issubdtype(arr_int.dtype, np.integer))
check('dtype_float', np.issubdtype(arr_float.dtype, np.floating))
check('float_uses_more_bytes', arr_float.nbytes >= arr_int.nbytes)

## Task 4: Views vs Copies
*Section: Indexing, Views & Copies*

**Interview Question:**  
Why does dtype selection matter in large ML pipelines?

Section 2 — Indexing, Views & Copies

Task 2.1: Views vs Copies

**Hints**
- # - Use slicing, not fancy indexing

In [ ]:
# Task 2.1
A = np.arange(20).reshape(10, 2)
A_slice = A[::2]  # basic slicing => view (usually)

A_slice[0, 0] = -999
print('A_slice:\n', A_slice)
print('A changed too (view):\n', A)

# Fancy indexing produces a copy
A_fancy = A[[0, 2, 4]]
A_fancy[0, 0] = 123456
print('Fancy indexing copy changed:', A_fancy[0, 0])
print('Original A unchanged at [0,0]:', A[0, 0])


In [ ]:
# Checks
check('slice_shape', A_slice.ndim == 2)
check('view_updates_parent', A[0, 0] == -999)
check('view_shares_memory', np.shares_memory(A, A_slice))
check('copy_shape', A_fancy.ndim == 2)
check('fancy_index_copy_no_parent_change', A[0, 0] != 123456)

## Task 5: Boolean masking
*Section: Boolean Masking*

Explain:
- Why did the original array change (or not)?

Section 2.2 — Boolean Masking

Task 2.2: Boolean masking

**Hints**
- # - Mean first

In [ ]:
rng = np.random.default_rng(0)
X = rng.standard_normal(1000)
x_mean = X.mean()
X_gt_mean = X[X > x_mean]

X_clipped = X.copy()
X_clipped[X_clipped < 0] = 0

print('mean:', x_mean)
print('above_mean count:', X_gt_mean.size)
print('negatives after replace:', (X_clipped < 0).sum())


In [ ]:
# Checks
check('x_shape', X.shape == (1000,))
check('mask_nonempty', X_gt_mean.size > 0)
check('gt_mean_condition', np.all(X_gt_mean > x_mean))
check('clip_non_negative', np.all(X_clipped >= 0))

## Task 6: Broadcasting Rules
*Section: Broadcasting*

Section 3 — Broadcasting

Task 3.1: Broadcasting Rules

**Hints**
- # - No reshape required
- # - Axis matters

In [ ]:
rng = np.random.default_rng(1)
A = rng.standard_normal((1000, 50))
b = rng.standard_normal((50,))

A_plus_b = A + b

# L2-normalize each row
def _safe_norm(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return np.maximum(n, 1e-12)

row_norms = _safe_norm(A)
A_norm = A / row_norms

print(A_plus_b.shape, A_norm.shape)
print('rows unit norm:', np.allclose(np.linalg.norm(A_norm, axis=1), 1.0, atol=1e-6))


In [ ]:
# Checks
check('A_shape', isinstance(locals().get('A', None), np.ndarray) and locals().get('A').shape == (1000, 50))
check('b_shape', isinstance(locals().get('b', None), np.ndarray) and locals().get('b').shape == (50,))
check('broadcast_shape', isinstance(locals().get('A_plus_b', None), np.ndarray) and locals().get('A_plus_b').shape == (1000, 50))
check('row_norm_shape', isinstance(locals().get('row_norms', None), np.ndarray) and locals().get('row_norms').shape in [(1000, 1), (1000,)])
check('A_norm_exists', isinstance(locals().get('A_norm', None), np.ndarray) and locals().get('A_norm').shape == (1000, 50))
check('rows_unit_norm', isinstance(locals().get('A_norm', None), np.ndarray) and np.allclose(np.linalg.norm(locals().get('A_norm'), axis=1), 1.0, atol=1e-5))

## Task 7: Fix a broadcasting trap
*Section: Broadcasting Trap*

Explain broadcasting step-by-step.

Section 3.2 — Broadcasting Trap

Task 3.2: Fix a broadcasting trap

In [ ]:
# Broadcasting trap
A = np.ones((3, 4))
error_raised = False
try:
    b = np.ones((5,))
    _ = A + b
except ValueError as e:
    error_raised = True
    print('Expected broadcasting error:', e)

# Fix with compatible shape
b = np.ones((4,))
fixed = A + b
print('Fixed shape:', fixed.shape)


In [ ]:
# Checks
check('broadcast_error_raised', bool(locals().get('error_raised', False)))
check('fixed_shape', isinstance(locals().get('fixed', None), np.ndarray) and locals().get('fixed').shape == (3, 4))

**Why this works.** Broadcasting is NumPy’s way of performing element-wise operations on arrays of different shapes without copying data.

It does this by pretending smaller arrays are expanded — using strides, not memory duplication.

## Task 8: Loop vs Vectorized
*Section: Vectorization vs Loops*

What was wrong with the original shapes?

Section 4 — Vectorization vs Loops

Task 4.1: Loop vs Vectorized

**Hints**
- # - Time it

In [ ]:
rng = np.random.default_rng(2)
X = rng.standard_normal(1_000_000)
mu = X.mean()
sigma = X.std()

# Loop
start = time.time()
X_loop = np.empty_like(X)
for i in range(X.shape[0]):
    X_loop[i] = (X[i] - mu) / sigma
loop_time = time.time() - start

# Vectorized
start = time.time()
X_vec = (X - mu) / sigma
vec_time = time.time() - start

print('loop_time:', round(loop_time, 4))
print('vec_time :', round(vec_time, 4))
print('speedup     :', round(loop_time / max(vec_time, 1e-12), 1), 'x')
print('allclose :', np.allclose(X_loop, X_vec))


In [ ]:
# Checks
check('same_shape', X_loop.shape == X_vec.shape == X.shape)
check('same_values', np.allclose(X_loop, X_vec, atol=1e-8))
check('vectorized_not_slower', vec_time <= loop_time * 1.2)

## Task 9: Pairwise Distance (FAANG Classic)
*Section: Vectorization vs Loops*

Why is vectorization faster?

Task 4.2: Pairwise Distance (FAANG Classic)

**Hints**
- # - Use (x - y)^2 expansion

In [ ]:
# Pairwise distance (FAANG classic)
def pairwise_distance(X):
    # ||x-y||^2 = ||x||^2 + ||y||^2 - 2 x·y
    sq = np.sum(X * X, axis=1, keepdims=True)
    dist2 = sq + sq.T - 2 * (X @ X.T)
    dist2 = np.maximum(dist2, 0.0)
    np.fill_diagonal(dist2, 0.0)
    return np.sqrt(dist2)

rng = np.random.default_rng(3)
X_test = rng.standard_normal((20, 5))
D = pairwise_distance(X_test)
print(D.shape)
print('diag ~0:', np.allclose(np.diag(D), 0, atol=1e-7))
print('symmetric:', np.allclose(D, D.T, atol=1e-7))


In [ ]:
# Checks
check('D_square', D.shape == (20, 20))
check('D_diag_zero', np.allclose(np.diag(D), 0.0, atol=1e-8))
check('D_symmetric', np.allclose(D, D.T, atol=1e-8))
check('D_nonnegative', np.all(D >= -1e-9))

## Task 10: Softmax
*Section: Numerical Stability*

Section 5 — Numerical Stability

Task 5.1: Softmax

Softmax converts logits into probabilities:

$$
\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

Stable form (subtract max logit in each row):

$$
\mathrm{softmax}(z)_i = \frac{e^{z_i - \max(z)}}{\sum_{j=1}^{K} e^{z_j - \max(z)}}
$$

**Hints**
- # - Subtract max per row

In [ ]:
def softmax_naive(X):
    ex = np.exp(X)
    return ex / ex.sum(axis=1, keepdims=True)

def softmax_stable(X):
    Z = X - X.max(axis=1, keepdims=True)
    ex = np.exp(Z)
    return ex / ex.sum(axis=1, keepdims=True)

rng = np.random.default_rng(9)
X_small = rng.normal(size=(4, 6))
X_big = X_small * 200

P1 = softmax_naive(X_small)
P2 = softmax_stable(X_small)
P3 = softmax_stable(X_big)

print('small rowsum naive:', P1.sum(axis=1))
print('small rowsum stable:', P2.sum(axis=1))
print('stable finite on large logits:', np.isfinite(P3).all())


In [ ]:
# Checks
check('naive_rows_sum_1', np.allclose(P1.sum(axis=1), 1.0, atol=1e-6))
check('stable_rows_sum_1', np.allclose(P2.sum(axis=1), 1.0, atol=1e-6))
check('stable_matches_naive_small', np.allclose(P1, P2, atol=1e-6))
check('stable_finite_large', np.isfinite(P3).all())

## Task 11: Matrix Multiplication
*Section: Linear Algebra*

Why does subtracting max work?

Section 6 — Linear Algebra

Task 6.1: Matrix Multiplication

In [ ]:
A = np.arange(6).reshape(2, 3)
B = np.arange(12).reshape(3, 4)
C = A @ B
print('C shape:', C.shape)

invalid_raised = False
try:
    _ = B @ A
except ValueError as e:
    invalid_raised = True
    print('Expected invalid matmul:', e)


In [ ]:
# Checks
check('valid_shape', C.shape == (2, 4))
check('invalid_raised', invalid_raised)

## Task 12: Solving Linear Systems
*Section: Linear Algebra*

Explain difference between dot, @, and matmul.

Task 6.2: Solving Linear Systems

In [ ]:
# Solving Ax=b
rng = np.random.default_rng(11)
A = rng.normal(size=(5, 5))
A += np.eye(5) * 0.5
b = rng.normal(size=(5,))

x = np.linalg.solve(A, b)
residual = np.linalg.norm(A @ x - b)
print('residual:', residual)


In [ ]:
# Checks
check('x_shape', x.shape == (5,))
check('small_residual', residual < 1e-6)

## Task 13: In-Place Operations
*Section: Performance & Memory*

Section 7 — Performance & Memory

Task 7.1: In-Place Operations

In [ ]:
# In-place vs out-of-place
X = np.arange(10.0)
Y = X

X_in = X.copy()
X_in += 1

X_out = X.copy()
X_out = X_out + 1

print('alias shares memory:', np.shares_memory(X, Y))
print('in-place result:', X_in)
print('out-of-place result:', X_out)


In [ ]:
# Checks
check('in_place_aliases', np.shares_memory(X, Y))
check('in_place_values', np.allclose(X_in, np.arange(10.0) + 1))
check('out_place_values', np.allclose(X_out, np.arange(10.0) + 1))

## Task 14: Strides
*Section: Performance & Memory*

Task 7.2: Strides

In [ ]:
# Strides
A = np.arange(20).reshape(10, 2)
A_slice = A[::2]

print('A.strides:', A.strides)
print('A_slice.strides:', A_slice.strides)


In [ ]:
# Checks
check('A_2d', A.ndim == 2)
check('strides_tuple', isinstance(A.strides, tuple) and len(A.strides) == 2)
check('slice_is_view', np.shares_memory(A, A_slice))

## Task 15: Mini case study (NumPy PCA)
*Section: Mini Case Study*

Section 8 — Mini Case Study

Task 8.1: Mini case study (NumPy PCA)

![Eigen Decomposition Overview](/images/eigen-decomposition-overview.png)

![Singular Value Decomposition (SVD)](/images/svd-overview.png)

In [ ]:
rng = np.random.default_rng(8)
X = rng.normal(size=(2000, 40))

# standardize
Xc = (X - X.mean(axis=0, keepdims=True)) / np.maximum(X.std(axis=0, keepdims=True), 1e-12)

# covariance
C = (Xc.T @ Xc) / (Xc.shape[0] - 1)

# eigendecomposition
evals, evecs = np.linalg.eigh(C)

k = 5
W = evecs[:, np.argsort(evals)[-k:][::-1]]
Z = Xc @ W

print('C shape:', C.shape)
print('W shape:', W.shape)
print('Z shape:', Z.shape)


In [ ]:
# Checks
check('cov_shape', C.shape == (40, 40))
check('W_shape', W.shape == (40, k))
check('Z_shape', Z.shape == (2000, k))
check('W_orthonormal', np.allclose(W.T @ W, np.eye(k), atol=1e-4))

**Why this works.** ## Section 8 — Mini Case Study (ML-Oriented PCA without sklearn)

### Task 8.1: Mini case study (NumPy PCA)